# Uncertainty-aware mangrove area estimate and REDD+ valuation

Companion code for the area estimate in Section 4.4 and the supplementary area-estimation methodology of the manuscript "Accuracy is not certainty" (Environmental Research Letters). For the Sundarbans study site it computes:

1. A design-based stratified area estimate (Olofsson et al. 2014) over the four probability bins of the final stacked-generalization map (random forest with feature pass-through), using 400 human-interpreted reference points.
2. An ensemble-spread interval across the five base learners and the four stacking configurations, as a complementary diagnostic.
3. Interpreter-threshold sensitivity (0.4 / 0.5 / 0.6).
4. The carbon stock and REDD+ value implied by the area estimate (manuscript Table 4).

## Inputs
- Prediction rasters (GeoTIFF) for the five base learners and four stacking configurations. Set `MODELS_DIR` in the config cell to their location. These are large and are not shipped here.
- Six Google Earth Engine FeatureCollections of interpreter scores: `projects/ee-islamkm/assets/interpreter{1,2,3}_200pts` (disagreement stratum) and `..._200pts_certain` (consensus stratum). Requires a one-time `earthengine authenticate` and read access to that project.

## Outputs (written to `outputs/`)
`bin_counts_final_model.csv`, `reference_points_with_probs.csv`, `olofsson_table.csv`, `ensemble_areas.csv`, `stacking_variant_areas.csv`, `interpreter_threshold_sensitivity.csv`, `carbon_redd_table.csv`, `area_estimate_summary.md`.

## Reproducing without GEE or the rasters
The two inputs that require GEE and the rasters (`reference_points_with_probs.csv` and `bin_counts_final_model.csv`) are shipped in `outputs/`. Set `REPRODUCE_FROM_CSV = True` in the config cell, then run the final "Quick verification" cell to recompute the Olofsson estimate and the carbon/REDD table from those two files alone, with no GEE access or large rasters required.

## 1. Setup

In [ ]:
import ee
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import math
from pathlib import Path

# ---- Configuration: edit these for your environment -------------------------
# Folder holding the nine prediction GeoTIFFs (5 base learners + 4 stacking
# configurations). These are large and are not shipped in this repo; point this
# at your local copy or a download. Only needed for a full run from rasters.
MODELS_DIR = Path('model_outputs')
# Results are written to / read from here. Shipped CSVs live in this folder so the
# headline numbers can be verified offline (see the Quick verification cell).
OUT_DIR = Path('outputs')
OUT_DIR.mkdir(parents=True, exist_ok=True)
# Set True to skip GEE and raster reads and verify from the shipped CSVs instead.
REPRODUCE_FROM_CSV = False
# -----------------------------------------------------------------------------

if not REPRODUCE_FROM_CSV:
    ee.Initialize()  # requires a one-time: earthengine authenticate

# Final model = random-forest stacking with feature pass-through (manuscript Figure 8).
FINAL_MODEL = MODELS_DIR / 'stacking_rf_pt_prediction.tif'
BASE_LEARNERS = {
    'knn':    MODELS_DIR / 'baselearner_knn_prediction.tif',
    'logreg': MODELS_DIR / 'baselearner_logreg_prediction.tif',
    'rf':     MODELS_DIR / 'baselearner_rf_prediction.tif',
    'svc':    MODELS_DIR / 'baselearner_svc_prediction.tif',
    'xgb':    MODELS_DIR / 'baselearner_xgb_prediction.tif',
}
STACKING_VARIANTS = {
    'logreg_npt': MODELS_DIR / 'stacking_logreg_npt_prediction.tif',
    'logreg_pt':  MODELS_DIR / 'stacking_logreg_pt_prediction.tif',
    'rf_npt':     MODELS_DIR / 'stacking_rf_npt_prediction.tif',
    'rf_pt':      MODELS_DIR / 'stacking_rf_pt_prediction.tif',
}

# Probability strata for the Olofsson estimator. Right edge slightly > 1 so that a
# pixel value of exactly 1.0 falls in the top bin.
BIN_EDGES = [0.0, 0.3, 0.5, 0.7, 1.0001]
BIN_LABELS = ['0.0-0.3', '0.3-0.5', '0.5-0.7', '0.7-1.0']

GEE_ASSETS = {
    'disagreement': [
        'projects/ee-islamkm/assets/interpreter1_200pts',
        'projects/ee-islamkm/assets/interpreter2_200pts',
        'projects/ee-islamkm/assets/interpreter3_200pts',
    ],
    'consensus': [
        'projects/ee-islamkm/assets/interpreter1_200pts_certain',
        'projects/ee-islamkm/assets/interpreter2_200pts_certain',
        'projects/ee-islamkm/assets/interpreter3_200pts_certain',
    ],
}

# ---- Carbon and REDD+ constants (used in the carbon valuation cell) ----------
# Ecosystem-carbon density (t C / ha): midpoint of the Sundarbans range reported by
# Rahman et al. (2015), Wetlands Ecology and Management 23:269-283 (159.5 to 360.0
# across vegetation types; 170.1 to 336.1 across salinity zones).
DENSITY_TC_HA = 260.0
# CO2-to-C molar mass ratio (44 / 12); standard IPCC stoichiometric conversion.
CO2_PER_C = 44.0 / 12.0
# REDD+ carbon-price scenarios (USD per t CO2e): $5 approximates the voluntary
# forest-carbon market floor (Forest Trends 2024); $50 a compliance-market level
# (World Bank 2024, State and Trends of Carbon Pricing); $15 is an intermediate tier.
PRICE_TIERS = [5, 15, 50]

## 2. Raster diagnostics

Confirm CRS, pixel area, and total pixel count for the final model before computing anything. Cross-check against manuscript Table 2 (237,963,203 total pixels).

In [ ]:
import math

with rasterio.open(FINAL_MODEL) as src:
    print('CRS:', src.crs)
    print('Transform:', src.transform)
    print('Width, Height:', src.width, src.height)
    print('Total pixels (raster grid, before masking):', src.width * src.height)
    px_w = abs(src.transform.a)
    px_h = abs(src.transform.e)
    bounds = src.bounds
    print(f'Bounds: {bounds}')
    if src.crs and src.crs.is_geographic:
        # Convert degrees to meters at the raster centroid latitude (Sundarbans, ~22 deg N)
        lat_center = (bounds.top + bounds.bottom) / 2.0
        meters_per_deg_lon = 111320.0 * math.cos(math.radians(lat_center))
        meters_per_deg_lat = 110574.0  # mean over the lat range; ~0.6% variation across the bbox
        px_w_m = px_w * meters_per_deg_lon
        px_h_m = px_h * meters_per_deg_lat
        PIXEL_AREA_M2 = px_w_m * px_h_m
        print(f'CRS is geographic. Using centroid latitude = {lat_center:.4f} deg N for the degree-to-meter conversion.')
        print(f'Pixel size in degrees: {px_w:.8f} x {px_h:.8f}')
        print(f'Pixel size in meters at centroid lat: {px_w_m:.4f} x {px_h_m:.4f}')
        print(f'Pixel area: {PIXEL_AREA_M2:.4f} m^2 (= {PIXEL_AREA_M2/10000:.6f} ha)')
    else:
        PIXEL_AREA_M2 = px_w * px_h
        print(f'Pixel size in raster CRS units: {px_w} x {px_h}')
        print(f'Pixel area: {PIXEL_AREA_M2:.4f} m^2 (assumes projected meters)')
    print('Nodata:', src.nodata)
    print('Dtype:', src.dtypes[0])

## 3. Per-bin pixel counts from the final model

Stream the raster in windows to count pixels per probability bin. Cross-check against Table 2.

In [ ]:
def count_pixels_by_bin(raster_path, bin_edges, valid_range=(0.0, 1.0)):
    counts = np.zeros(len(bin_edges) - 1, dtype=np.int64)
    n_valid = 0
    n_nodata = 0
    with rasterio.open(raster_path) as src:
        nd = src.nodata
        for ji, window in src.block_windows(1):
            arr = src.read(1, window=window)
            mask = np.isfinite(arr)
            if nd is not None:
                mask &= (arr != nd)
            mask &= (arr >= valid_range[0]) & (arr <= valid_range[1])
            n_valid  += int(mask.sum())
            n_nodata += int((~mask).sum())
            vals = arr[mask]
            h, _ = np.histogram(vals, bins=bin_edges)
            counts += h
    return counts, n_valid, n_nodata

final_counts, n_valid, n_nodata = count_pixels_by_bin(FINAL_MODEL, BIN_EDGES)
df_bins = pd.DataFrame({
    'bin': BIN_LABELS,
    'pixel_count': final_counts,
    'pct_of_valid': 100.0 * final_counts / n_valid,
})
print('Total valid pixels:', n_valid)
print('Total nodata/invalid:', n_nodata)
print(df_bins)
df_bins.to_csv(OUT_DIR / 'bin_counts_final_model.csv', index=False)

## 4. Pull reference points from GEE

Six FeatureCollections: three interpreters × two strata. Same 200 points per stratum across interpreters.

In [ ]:
from shapely.geometry import shape as shp_shape

def fc_to_gdf(asset_id):
    """Pull an EE FeatureCollection to a GeoDataFrame.
    - Point features: kept as one row with the parent properties.
    - MultiPoint features: expanded to one row per child point; each child inherits the parent's properties
      (this is the structure of the consensus assets, where points are grouped by val_int score level).
    - Null or other geometry types: skipped, counted, and reported."""
    fc = ee.FeatureCollection(asset_id)
    info = fc.getInfo()
    feats = info['features']
    rows = []
    geoms = []
    prop_keys_seen = set()
    geom_types_seen = {}
    n_expanded = 0
    n_skipped = 0
    for ft in feats:
        props = ft.get('properties', {}) or {}
        prop_keys_seen.update(props.keys())
        geom_obj = ft.get('geometry')
        if geom_obj is None:
            n_skipped += 1
            continue
        gtype = geom_obj.get('type')
        geom_types_seen[gtype] = geom_types_seen.get(gtype, 0) + 1
        try:
            g = shp_shape(geom_obj)
        except Exception:
            n_skipped += 1
            continue
        if g.geom_type == 'Point':
            rows.append({**props, '_lon': float(g.x), '_lat': float(g.y)})
            geoms.append(g)
        elif g.geom_type == 'MultiPoint':
            for pt in g.geoms:
                rows.append({**props, '_lon': float(pt.x), '_lat': float(pt.y)})
                geoms.append(pt)
                n_expanded += 1
        else:
            n_skipped += 1
    gdf = gpd.GeoDataFrame(rows, geometry=geoms, crs='EPSG:4326')
    return gdf, sorted(prop_keys_seen), geom_types_seen, n_skipped, n_expanded

all_pts = []
for stratum, assets in GEE_ASSETS.items():
    for i, asset_id in enumerate(assets, start=1):
        gdf, prop_keys, geom_types, skipped, expanded = fc_to_gdf(asset_id)
        print(f'{stratum} interp{i}: {len(gdf)} rows (after expansion), props={prop_keys}, geom_types={geom_types}, skipped={skipped}, expanded_from_multipoint={expanded}')
        gdf['_stratum'] = stratum
        gdf['_interpreter'] = i
        all_pts.append(gdf)
pts = pd.concat(all_pts, ignore_index=True)
print(f'\nTotal rows: {len(pts)}; per-stratum counts:')
print(pts.groupby(['_stratum','_interpreter']).size())
pts.head()

The interpreter score is stored in the `val_int` property of each feature (confirmed by the property keys printed above). The next cell projects the points to UTM, clusters points within 10 m of each other so the same physical location scored by the three interpreters is treated as one reference point, and pivots to one row per reference point with the three interpreter scores.

In [ ]:
from pyproj import Transformer
from shapely.strtree import STRtree
from shapely.geometry import Point

SCORE_FIELD = 'val_int'
BUFFER_M = 10.0  # points within this distance of each other are treated as the same physical reference point

# Project lon/lat to UTM Zone 45N (Sundarbans). Slight distortion at ~90 deg E is well within 10 m.
to_utm  = Transformer.from_crs('EPSG:4326', 'EPSG:32645', always_xy=True)
to_geog = Transformer.from_crs('EPSG:32645', 'EPSG:4326', always_xy=True)
pts['_utm_x'], pts['_utm_y'] = to_utm.transform(pts['_lon'].values, pts['_lat'].values)

def union_find_cluster(xs, ys, buffer_m):
    """Group points so any two within buffer_m of each other share a cluster id."""
    n = len(xs)
    parent = list(range(n))
    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i
    def union(i, j):
        ri, rj = find(i), find(j)
        if ri != rj:
            parent[ri] = rj
    geoms = [Point(x, y) for x, y in zip(xs, ys)]
    tree = STRtree(geoms)
    for i, g in enumerate(geoms):
        for j in tree.query(g.buffer(buffer_m)):
            j = int(j)
            if j != i and g.distance(geoms[j]) <= buffer_m:
                union(i, j)
    return [find(i) for i in range(n)]

# Cluster within each stratum so disagreement and consensus reference points never merge
pts['_cluster_id'] = ''
for stratum in pts['_stratum'].unique():
    mask = pts['_stratum'] == stratum
    sub = pts[mask]
    ids = union_find_cluster(sub['_utm_x'].values, sub['_utm_y'].values, BUFFER_M)
    pts.loc[mask, '_cluster_id'] = [f'{stratum}_{i}' for i in ids]

print('Unique cluster ids per stratum:')
print(pts.groupby('_stratum')['_cluster_id'].nunique())

cluster_size = pts.groupby(['_stratum', '_cluster_id']).size().reset_index(name='n_rows')
print('\nDistribution of rows-per-cluster (3 == all three interpreters matched; <3 means buffer missed):')
print(cluster_size['n_rows'].value_counts().sort_index())

# Pivot wide: one row per cluster (= one logical reference point), interpreter scores in three columns
wide = pts.pivot_table(index=['_stratum', '_cluster_id'], columns='_interpreter', values=SCORE_FIELD, aggfunc='mean').reset_index()
wide.columns = ['_stratum', '_cluster_id', 'score1', 'score2', 'score3']

# Representative location per cluster = centroid in UTM, transformed back to WGS84 for raster sampling
centroids = pts.groupby(['_stratum', '_cluster_id']).agg(_utm_x=('_utm_x','mean'), _utm_y=('_utm_y','mean')).reset_index()
centroids['lon'], centroids['lat'] = to_geog.transform(centroids['_utm_x'].values, centroids['_utm_y'].values)
wide = wide.merge(centroids[['_stratum', '_cluster_id', 'lon', 'lat']], on=['_stratum', '_cluster_id'])

wide['score_mean']     = wide[['score1', 'score2', 'score3']].mean(axis=1)
wide['score_std']      = wide[['score1', 'score2', 'score3']].std(axis=1, ddof=1)
wide['n_interpreters'] = wide[['score1', 'score2', 'score3']].notna().sum(axis=1)
wide['ref_label']      = (wide['score_mean'] >= 0.5).astype(int)
print('\nFinal wide table shape:', wide.shape)
print('Per-stratum counts:')
print(wide.groupby('_stratum').size())
wide.head()

## 5. Extract final-model probability at each reference point

Sample the final stacked-probability raster at each reference point and assign to a probability bin.

In [ ]:
def sample_raster_at_lonlat(raster_path, lons, lats, src_crs='EPSG:4326'):
    with rasterio.open(raster_path) as src:
        if src.crs.to_string() != src_crs:
            from pyproj import Transformer
            t = Transformer.from_crs(src_crs, src.crs, always_xy=True)
            xs, ys = t.transform(list(lons), list(lats))
        else:
            xs, ys = list(lons), list(lats)
        vals = [v[0] for v in src.sample(list(zip(xs, ys)))]
    return np.array(vals, dtype=float)

wide['prob_final'] = sample_raster_at_lonlat(FINAL_MODEL, wide['lon'], wide['lat'])
wide['prob_bin'] = pd.cut(wide['prob_final'], bins=BIN_EDGES, labels=BIN_LABELS, include_lowest=True, right=False)
print(wide.groupby('prob_bin', observed=True)['ref_label'].agg(['count', 'mean']))
wide.to_csv(OUT_DIR / 'reference_points_with_probs.csv', index=False)

## 6. Olofsson stratified estimator (Method C)

In [ ]:
N_total = int(final_counts.sum())
W = final_counts / N_total

olofsson_rows = []
for i, lab in enumerate(BIN_LABELS):
    sub = wide[wide['prob_bin'] == lab]
    n_i = len(sub)
    if n_i > 0:
        ybar = sub['ref_label'].mean()
        s2   = sub['ref_label'].var(ddof=1) if n_i > 1 else 0.0
    else:
        ybar, s2 = np.nan, np.nan
    olofsson_rows.append({
        'bin': lab,
        'N_i': int(final_counts[i]),
        'W_i': float(W[i]),
        'n_i': n_i,
        'ybar_i': ybar,
        's2_i': s2,
    })
olof = pd.DataFrame(olofsson_rows)
print(olof)

if not olof['ybar_i'].isna().any():
    p_hat = float((olof['W_i'] * olof['ybar_i']).sum())
    var_p = float(((olof['W_i'] ** 2) * olof['s2_i'] / olof['n_i'].clip(lower=1)).sum())
    se_p  = np.sqrt(var_p)
    print(f'p_hat = {p_hat:.6f}')
    print(f'SE(p_hat) = {se_p:.6f}')
    print(f'95% CI on proportion: [{p_hat - 1.96*se_p:.6f}, {p_hat + 1.96*se_p:.6f}]')
else:
    print('At least one bin has zero reference points. Supplementary sampling needed.')

In [ ]:
# PIXEL_AREA_M2 was computed in section 2 (geographic-aware: degrees -> meters at centroid latitude)
A_total_ha = N_total * PIXEL_AREA_M2 / 10000.0
if not olof['ybar_i'].isna().any():
    A_hat_ha = p_hat * A_total_ha
    SE_A_ha  = se_p  * A_total_ha
    print(f'Total study-area area: {A_total_ha:,.1f} ha')
    print(f'Estimated mangrove area: {A_hat_ha:,.1f} ha (95% CI {A_hat_ha - 1.96*SE_A_ha:,.1f} - {A_hat_ha + 1.96*SE_A_ha:,.1f} ha)')
olof['contribution_ha']    = olof['W_i'] * olof['ybar_i'] * A_total_ha
olof['contribution_se_ha'] = (olof['W_i'] ** 2 * olof['s2_i'] / olof['n_i'].clip(lower=1)).pow(0.5) * A_total_ha
olof.to_csv(OUT_DIR / 'olofsson_table.csv', index=False)
olof

## 7. Ensemble-spread interval (Method D)

For each base learner, count pixels above 0.5 × pixel area. Report mean and range across the five base learners.

In [ ]:
def count_pixels_above_threshold(raster_path, thr=0.5, valid_range=(0.0, 1.0)):
    n_above = 0
    n_valid = 0
    with rasterio.open(raster_path) as src:
        nd = src.nodata
        for ji, window in src.block_windows(1):
            arr = src.read(1, window=window)
            mask = np.isfinite(arr)
            if nd is not None:
                mask &= (arr != nd)
            mask &= (arr >= valid_range[0]) & (arr <= valid_range[1])
            n_valid += int(mask.sum())
            n_above += int((arr[mask] >= thr).sum())
    return n_above, n_valid

ens_rows = []
for name, path in BASE_LEARNERS.items():
    above, valid = count_pixels_above_threshold(path)
    area_ha = above * PIXEL_AREA_M2 / 10000.0
    ens_rows.append({'model': name, 'pixels_above_0.5': above, 'valid_pixels': valid, 'area_ha': area_ha})
ens = pd.DataFrame(ens_rows)
print(ens)
print(f'Ensemble mean: {ens["area_ha"].mean():,.1f} ha')
print(f'Ensemble SD:   {ens["area_ha"].std(ddof=1):,.1f} ha')
print(f'Ensemble range: [{ens["area_ha"].min():,.1f}, {ens["area_ha"].max():,.1f}] ha')
ens.to_csv(OUT_DIR / 'ensemble_areas.csv', index=False)

## 8. Sensitivity checks

Interpreter-threshold sensitivity (0.4, 0.5, 0.6) and stacking-variant sensitivity (rerun ensemble across the four stacking configurations).

In [ ]:
sensit_rows = []
for thr in [0.4, 0.5, 0.6]:
    tmp = wide.copy()
    tmp['ref_label'] = (tmp['score_mean'] >= thr).astype(int)
    p = 0.0
    for i, lab in enumerate(BIN_LABELS):
        sub = tmp[tmp['prob_bin'] == lab]
        if len(sub) > 0:
            p += W[i] * sub['ref_label'].mean()
    sensit_rows.append({'interp_threshold': thr, 'p_hat': p, 'area_ha': p * A_total_ha})
sensit = pd.DataFrame(sensit_rows)
print(sensit)
sensit.to_csv(OUT_DIR / 'interpreter_threshold_sensitivity.csv', index=False)

In [ ]:
# Stacking variants (sensitivity check on Method D)
stack_rows = []
for name, path in STACKING_VARIANTS.items():
    above, valid = count_pixels_above_threshold(path)
    area_ha = above * PIXEL_AREA_M2 / 10000.0
    stack_rows.append({'model': name, 'pixels_above_0.5': above, 'valid_pixels': valid, 'area_ha': area_ha})
stack = pd.DataFrame(stack_rows)
print(stack)
stack.to_csv(OUT_DIR / 'stacking_variant_areas.csv', index=False)

## 9. Carbon stock and REDD+ valuation (Table 4)

Propagate the area estimate and its 95% confidence interval into ecosystem carbon stock, CO2 equivalent, and REDD+ value at three carbon-price tiers, using the constants defined in the config cell.

In [ ]:
# ---- Carbon stock and REDD+ value implied by the area estimate (Table 4) -----
# Constants DENSITY_TC_HA, CO2_PER_C and PRICE_TIERS are defined in the config cell.
# Carbon and CO2e are computed directly from area and rounded once for display
# (matching the manuscript convention); each REDD+ value = displayed CO2e * price.
area_central = A_hat_ha
area_lo = A_hat_ha - 1.96 * SE_A_ha
area_hi = A_hat_ha + 1.96 * SE_A_ha

def _carbon(area_ha):
    c_mt   = round(area_ha * DENSITY_TC_HA / 1e6, 1)               # ecosystem carbon, Mt C
    co2_mt = round(area_ha * DENSITY_TC_HA * CO2_PER_C / 1e6, 1)   # CO2 equivalent, Mt CO2e
    return c_mt, co2_mt

carbon_rows = []
for name, a in [('central', area_central), ('95% CI lower', area_lo), ('95% CI upper', area_hi)]:
    c_mt, co2_mt = _carbon(a)
    row = {'estimate': name, 'area_ha': round(a), 'carbon_Mt_C': c_mt, 'co2e_Mt': co2_mt}
    for p in PRICE_TIERS:
        row[f'redd_USD_M_at_${p}'] = round(co2_mt * p)
    carbon_rows.append(row)
carbon = pd.DataFrame(carbon_rows)
print(carbon.to_string(index=False))
carbon.to_csv(OUT_DIR / 'carbon_redd_table.csv', index=False)

## 10. Summary

In [ ]:
summary_lines = [
    '# R2_C01 area-estimation summary',
    '',
    f'Total study-area area: {A_total_ha:,.1f} ha',
    f'GMW v4 reference for Bangladesh mangroves (2020): ~425,000 ha (whole country, not just study area)',
    '',
    '## Olofsson stratified estimator',
    f'Estimated mangrove area: {A_hat_ha:,.1f} ha',
    f'95% CI: {A_hat_ha - 1.96*SE_A_ha:,.1f} - {A_hat_ha + 1.96*SE_A_ha:,.1f} ha',
    '',
    '## Ensemble-spread interval (base learners)',
    f'Mean: {ens["area_ha"].mean():,.1f} ha; SD: {ens["area_ha"].std(ddof=1):,.1f} ha; Range: [{ens["area_ha"].min():,.1f}, {ens["area_ha"].max():,.1f}] ha',
    '',
    '## Stacking-variant spread',
    f'Mean: {stack["area_ha"].mean():,.1f} ha; Range: [{stack["area_ha"].min():,.1f}, {stack["area_ha"].max():,.1f}] ha',
]
summary = '\n'.join(summary_lines)
(OUT_DIR / 'area_estimate_summary.md').write_text(summary, encoding='utf-8')
print(summary)

## 11. Quick verification from the shipped CSVs

This cell recomputes the headline Olofsson area estimate and the carbon/REDD table using only the two shipped CSV files, with no GEE access and no rasters. A reader can run this cell alone (set `REPRODUCE_FROM_CSV = True` in the config cell and run the config cell first) to confirm the numbers reported in the paper.

In [ ]:
# Self-contained verification from the two shipped CSVs (no GEE, no rasters).
ref  = pd.read_csv(OUT_DIR / 'reference_points_with_probs.csv')
binc = pd.read_csv(OUT_DIR / 'bin_counts_final_model.csv')

Ntot = int(binc['pixel_count'].sum())
Wv   = (binc.set_index('bin')['pixel_count'] / Ntot).to_dict()

# Pixel area (m^2) from the section-2 raster diagnostics; fallback value lets this
# cell run on its own when the raster has not been read (EPSG:4326, centroid-lat
# degree-to-meter conversion at the Sundarbans).
pa = PIXEL_AREA_M2 if 'PIXEL_AREA_M2' in dir() else 20.9458
A_total = Ntot * pa / 1e4

p_hat = 0.0
var_p = 0.0
for lab in BIN_LABELS:
    sub = ref[ref['prob_bin'] == lab]
    p_hat += Wv[lab] * sub['ref_label'].mean()
    var_p += Wv[lab] ** 2 * sub['ref_label'].var(ddof=1) / max(len(sub), 1)
se_p = var_p ** 0.5
A_hat = p_hat * A_total
SE_A  = se_p * A_total

print(f'Olofsson area  = {A_hat:,.0f} ha   95% CI [{A_hat - 1.96*SE_A:,.0f}, {A_hat + 1.96*SE_A:,.0f}]')
# Carbon and CO2e computed directly from area (matching the manuscript Table 4).
carbon_central = round(A_hat * DENSITY_TC_HA / 1e6, 1)
co2_central    = round(A_hat * DENSITY_TC_HA * CO2_PER_C / 1e6, 1)
print(f'Ecosystem carbon = {carbon_central:,.1f} Mt C   CO2e = {co2_central:,.1f} Mt')
for p in PRICE_TIERS:
    print(f'  REDD+ at ${p}/tCO2e = {round(co2_central * p):,} USD million (central)')